In [1]:
#Browsing indi table in DB for potential appendiceal cancer analyses.

import sqlite3
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parents[0]
db_path = project_root / "database" / "faers.db"
conn = sqlite3.connect(db_path)

indi = pd.read_sql_query("SELECT * FROM indi LIMIT 10000", conn)

In [ ]:
print(indi.shape)
print(indi.dtypes)

primaryid         int64
caseid            int64
indi_drug_seq     int64
indi_pt             str
source_quarter      str
dtype: object

In [ ]:
indi.isnull().sum()  

primaryid         0
caseid            0
indi_drug_seq     0
indi_pt           0
source_quarter    0
dtype: int64

In [5]:
indi.nunique()


primaryid         1322
caseid            1322
indi_drug_seq      592
indi_pt            570
source_quarter       1
dtype: int64

In [6]:
indi['indi_pt'].value_counts().head(20)


indi_pt
Product used for unknown indication         4746
Rheumatoid arthritis                         948
Attention deficit hyperactivity disorder     151
Asthma                                       129
Hypertension                                 123
Pain                                         106
HIV infection                                101
Autism spectrum disorder                      93
Bipolar disorder                              93
Narcolepsy                                    82
Schizophrenia                                 79
Depression                                    77
Prophylaxis                                   72
Oppositional defiant disorder                 64
Cataplexy                                     64
Coronary artery disease                       64
Juvenile idiopathic arthritis                 62
Affective disorder                            60
Parkinson's disease                           53
Sleep disorder                                50
Name: count,

In [12]:
cancer_indi_counts = pd.read_sql_query("""
SELECT indi_pt, COUNT(*) AS reports
FROM indi
WHERE indi_pt LIKE '%appendic%'
   OR indi_pt LIKE '%colorect%'
   OR indi_pt LIKE '%periton%'
   OR indi_pt LIKE '%pseudomyxoma%'
   OR indi_pt LIKE '%cecal%'
   OR indi_pt LIKE '%mucinous%'
   OR indi_pt LIKE '%colon cancer%'
GROUP BY indi_pt
ORDER BY reports DESC
""", conn)

cancer_indi_counts



,indi_pt,reports
0,Colon cancer,6647
1,Colorectal cancer metastatic,5273
2,Colorectal cancer,3094
3,Colon cancer metastatic,1236
4,Malignant peritoneal neoplasm,736
...,...,...
66,Complicated appendicitis,2
67,Borderline mucinous tumour of ovary,2
68,Peritoneal lavage,1
69,Appendicitis noninfective,1


In [14]:
print(cancer_indi_counts.to_string())


                                                   indi_pt  reports
0                                             Colon cancer     6647
1                             Colorectal cancer metastatic     5273
2                                        Colorectal cancer     3094
3                                  Colon cancer metastatic     1236
4                            Malignant peritoneal neoplasm      736
5                                 Metastases to peritoneum      605
6                                      Peritoneal dialysis      575
7                                Colorectal adenocarcinoma      460
8                                              Peritonitis      358
9                                    Colon cancer stage IV      310
10                             Colorectal cancer stage III      234
11                                  Colon cancer stage III      171
12                                   Peritonitis bacterial      138
13                         HER2 positive colorec

In [ ]:
cancer_terms = [
    'Colon cancer',
    'Colorectal cancer metastatic',
    'Colorectal cancer',
    'Colon cancer metastatic',
    'Malignant peritoneal neoplasm',
    'Metastases to peritoneum',
    'Colorectal adenocarcinoma',
    'Colon cancer stage IV',
    'Colorectal cancer stage III',
    'Colon cancer stage III',
    'HER2 positive colorectal cancer',
    'Colorectal cancer stage IV',
    'Peritoneal mesothelioma malignant',
    'Colon cancer stage II',
    'Colon cancer recurrent',
    'Colorectal cancer stage II',
    'Colorectal neoplasm',
    'Peritoneal sarcoma',
    'Colorectal cancer recurrent',
    'Peritoneal neoplasm',
    'Mucinous adenocarcinoma of appendix',
    'Peritoneal carcinoma metastatic',
    'Pseudomyxoma peritonei',
    'Peritoneal mesothelioma malignant recurrent',
    'Intraductal papillary mucinous neoplasm',
    'Colorectal cancer stage I',
    'Colon cancer stage I',
    'Appendiceal mucinous neoplasm',
    'Hereditary non-polyposis colorectal cancer syndrome',
]

cancer_drugs = pd.read_sql_query(f"""
SELECT 
    d.drugname,
    COUNT(DISTINCT d.primaryid) AS reports
FROM drug d
JOIN indi i ON d.primaryid = i.primaryid
WHERE i.indi_pt IN ({','.join(['?' for _ in cancer_terms])})
GROUP BY d.drugname
ORDER BY reports DESC
LIMIT 30
""", conn, params=cancer_terms)

cancer_drugs
